# Machine Learning - Practical 06: Cross-validation, Tuning, Pipelines and Leakage

**Course:** Machine Learning - National University of Kyiv-Mohyla Academy (NaUKMA)

**Instructor:** Dmytro Kuzmenko - kuzmenko@ukma.edu.ua

| | |
|---|---|
| Week | 6 |
| Module | 6. Evaluation Done Properly |
| Format | Practical session (not graded - exam preparation) |
| Estimated time | 1.5-2 h of active work including discussion |
| Prerequisites | P01-P05 |


## AI Use Disclosure

Fill this in before submitting (see course policy).

| Field | Your entry |
|---|---|
| AI tools used | |
| Nature of assistance | |
| Representative prompts or relevant interaction | |
| What I independently verified or changed | |

## Learning Objectives

- Choose a cross-validation scheme that matches the data structure (i.i.d., grouped, time-ordered, imbalanced).
- Distinguish valid and leaking preprocessing inside cross-validation and quantify the optimism that leakage causes.
- Use GridSearchCV with a Pipeline so that every preprocessing step is refit on training folds only.

## Warm-up (10 min)

### Question 1 (multiple choice)

A pipeline applies PCA, and then the transformed data are split into folds for cross-validation. "PCA was fit on the full dataset before cross-validation." What is the methodological problem with this step?

- A. PCA is unsupervised, so it can never cause leakage.
- B. The PCA transformation was estimated using the validation folds as well as the training folds, so information about the validation folds influences the transformed training data (leakage).
- C. PCA makes cross-validation slower; that is the only issue.
- D. Nothing - PCA must always be fit on the full dataset.


**Your answer:**

### Question 2 (mini-interpretation)

Two models were tuned with the same protocol:

- Model A: train F1 = 0.98, validation F1 = 0.61
- Model B: train F1 = 0.82, validation F1 = 0.79

Which model would you prefer, and what further evidence would you collect before trusting that choice?


**Your answer:**

### Question 3 (quick reasoning)

A dataset has 400 rows and a 3% positive class. Plain KFold(5) is used for model selection. What can go wrong, and which splitter fixes it?


**Your answer:**

## Guided Exercise (60 min)

We compare validation schemes on three synthetic problems and quantify the cost of leakage. All experiments are deterministic (fixed seeds).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import (KFold, StratifiedKFold, GroupKFold,
                                     TimeSeriesSplit, GridSearchCV, cross_val_score)
%matplotlib inline
np.random.seed(42)

### Task 1: KFold vs StratifiedKFold on an imbalanced problem

**Experiment.** Build a small imbalanced classification set (200 samples, ~3% positives) and inspect how many positive samples land in each fold under plain KFold and under StratifiedKFold.

In [2]:
X, y = make_classification(n_samples=200, n_features=12, n_informative=6, n_redundant=2,
                           weights=[0.97, 0.03], n_clusters_per_class=1,
                           flip_y=0.02, random_state=1)
print("Overall positive rate: {:.3f}".format(y.mean()))

rows = []
for name, splitter in [("KFold", KFold(5, shuffle=True, random_state=42)),
                       ("StratifiedKFold", StratifiedKFold(5, shuffle=True, random_state=42))]:
    for fold, (tr, te) in enumerate(splitter.split(X, y)):
        rows.append({"splitter": name, "fold": fold,
                     "n_test": len(te),
                     "pos_in_train": int((y[tr] == 1).sum()),
                     "pos_in_test": int((y[te] == 1).sum())})
fold_stats = pd.DataFrame(rows)
print(fold_stats.to_string(index=False))

Overall positive rate: 0.035
       splitter  fold  n_test  pos_in_train  pos_in_test
          KFold     0      40             6            1
          KFold     1      40             5            2
          KFold     2      40             7            0
          KFold     3      40             4            3
          KFold     4      40             6            1
StratifiedKFold     0      40             6            1
StratifiedKFold     1      40             6            1
StratifiedKFold     2      40             6            1
StratifiedKFold     3      40             5            2
StratifiedKFold     4      40             5            2


**Decision.** Which splitter would you use for this data, and why?


**Your answer:**

**Evidence.** What exactly in the table above is the evidence for your decision?


**Your answer:**

**Interpretation.** What is the practical consequence of a fold with zero (or one) positive samples, both for training and for evaluating the model?


**Your answer:**

**Counterfactual.** Predict what changes if (a) the number of folds is increased to 10, or (b) the positive rate drops to 0.5%. Which splitter becomes even more necessary?


**Your answer:**

### Task 2: Time-ordered data - KFold, GroupKFold, TimeSeriesSplit

**Experiment.** Generate a synthetic series y_t = sin(2*pi*t/50) + 0.02*t + noise, with features [t/N, (t/N)^2]. Compare three validation schemes:

- KFold with shuffling (ignores order);
- GroupKFold over chunks of 20 consecutive points (respects grouping, ignores order);
- TimeSeriesSplit (expanding window, strictly past-to-future).

In [3]:
N = 600
t = np.arange(N)
rng = np.random.RandomState(42)
y_series = np.sin(2 * np.pi * t / 50) + 0.02 * t + rng.normal(0, 0.08, N)
X_series = np.column_stack([t / N, (t / N) ** 2])
groups = t // 20

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(t, y_series, lw=1)
ax.set_xlabel("time index t")
ax.set_ylabel("y_t")
ax.set_title("Synthetic time series: trend + seasonal component + noise")
plt.tight_layout()

In [4]:
schemes = {
    "KFold (shuffled)": KFold(5, shuffle=True, random_state=42),
    "GroupKFold (time chunks)": GroupKFold(n_splits=5),
    "TimeSeriesSplit": TimeSeriesSplit(n_splits=5),
}
cv_results = {}
for name, splitter in schemes.items():
    if name.startswith("GroupKFold"):
        cv_results[name] = cross_val_score(Ridge(), X_series, y_series, cv=splitter,
                                           groups=groups, scoring="r2")
    else:
        cv_results[name] = cross_val_score(Ridge(), X_series, y_series, cv=splitter, scoring="r2")

summary = pd.DataFrame(cv_results).T
summary.columns = ["fold {}".format(i + 1) for i in range(5)]
summary["mean R2"] = summary.mean(axis=1)
print(summary.round(3).to_string())

fig, ax = plt.subplots(figsize=(6.5, 3.5))
means = [v.mean() for v in cv_results.values()]
ax.bar(cv_results.keys(), means, color=["#4C72B0", "#DD8452", "#C44E52"])
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("mean CV R2")
ax.set_title("Mean CV R2 by validation scheme (Ridge on [t, t^2])")
plt.tight_layout()

                          fold 1  fold 2  fold 3  fold 4  fold 5  mean R2
KFold (shuffled)           0.952   0.956   0.954   0.956   0.953    0.954
GroupKFold (time chunks)   0.921   0.902   0.945   0.964   0.940    0.934
TimeSeriesSplit           -5.978  -2.215  -0.176  -0.176  -0.336   -1.776


**Decision.** Which scheme gives the honest estimate of how the model will behave on future data, and why?


**Your answer:**

**Interpretation.** Why does the shuffled KFold look so good (R2 near 0.95) while TimeSeriesSplit collapses to a negative R2? Where does the "extra" information come from?


**Your answer:**

**Counterfactual.** Suppose we add sin(2*pi*t/50) and cos(2*pi*t/50) as explicit features. Predict how the TimeSeriesSplit score would change, and explain the mechanism.


**Your answer:**

### Task 3: GridSearchCV inside a Pipeline vs leaking preprocessing

**Experiment.** Use a wide synthetic set (120 samples, 200 features, only 4 informative). Tune LogisticRegression C in two ways:

- Leaky: StandardScaler + SelectKBest are fit on the **full** dataset, then GridSearchCV runs on the already transformed matrix (scaling and feature selection never see the CV folds).
- Correct: StandardScaler + SelectKBest + LogisticRegression are wrapped in a Pipeline and GridSearchCV refits the whole pipeline inside every training fold.

In [5]:
X, y = make_classification(n_samples=120, n_features=200, n_informative=4, n_redundant=10,
                           n_clusters_per_class=1, flip_y=0.12, random_state=42)

cv5 = StratifiedKFold(5, shuffle=True, random_state=42)

# --- LEAKY: preprocessing fit once on the full dataset ---
scaler_full = StandardScaler().fit(X)
X_scaled = scaler_full.transform(X)
selector_full = SelectKBest(f_classif, k=25).fit(X_scaled, y)
X_selected = selector_full.transform(X_scaled)
gs_leaky = GridSearchCV(LogisticRegression(max_iter=3000), {"C": [0.01, 0.1, 1.0, 10.0]},
                        cv=cv5).fit(X_selected, y)

# --- CORRECT: preprocessing inside the pipeline ---
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(f_classif, k=25)),
    ("lr", LogisticRegression(max_iter=3000)),
])
gs_correct = GridSearchCV(pipe, {"lr__C": [0.01, 0.1, 1.0, 10.0]}, cv=cv5).fit(X, y)

print("LEAKY   best_score_ = {:.4f}   best C = {}".format(gs_leaky.best_score_, gs_leaky.best_params_))
print("CORRECT best_score_ = {:.4f}   best C = {}".format(gs_correct.best_score_, gs_correct.best_params_))

LEAKY   best_score_ = 0.8333   best C = {'C': 0.1}
CORRECT best_score_ = 0.7667   best C = {'lr__C': 10.0}


In [6]:
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.bar(["leaky (preprocessing on full data)", "correct (pipeline inside CV)"],
       [gs_leaky.best_score_, gs_correct.best_score_],
       color=["#C44E52", "#55A868"])
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("best CV score")
ax.set_title("GridSearchCV: leaking vs correct preprocessing")
for i, v in enumerate([gs_leaky.best_score_, gs_correct.best_score_]):
    ax.text(i, v + 0.01, "{:.3f}".format(v), ha="center")
plt.tight_layout()

**Decision.** Which number would you report in a paper, and why is the other one optimistic?


**Your answer:**

**Evidence.** Besides the mean score, what else differs between the two runs, and why is that itself evidence of a methodological problem?


**Your answer:**

**Interpretation.** Explain the mechanism: how does fitting SelectKBest on the full dataset (which includes every validation fold) inflate the CV estimate?


**Your answer:**

**Counterfactual.** Predict what happens to the gap if we (a) increase the number of noise features from 200 to 400, or (b) decrease k in SelectKBest from 25 to 5. Which direction makes selection more unstable, and hence leakage more damaging?


**Your answer:**

## Discussion Questions (15 min)

1. Medical data contain several measurements per patient, and patients must never appear in both train and validation folds. Which splitter do you use, and why does plain KFold overstate performance?
2. Why is it not enough to shuffle rows for time-series data? What structure does the shuffle destroy, and which splitter respects it?
3. GridSearchCV reports the best score from the tuning folds. Why is that score an optimistic estimate for a tuned model, and what protocol gives an unbiased estimate (nested CV)?
4. Some preprocessing leaks are dangerous only when they see the target (feature selection, oversampling), others are dangerous even when unsupervised (scaling or imputation fit on full data when folds have different distributions). Give one example of each from today's experiments.
5. A colleague says "I only looked at the test set once". Why is even a single use of the test set for a decision a problem?
6. When, if ever, is it legitimate to fit preprocessing on the full dataset before splitting?

## Challenge (25 min)

### Task 4: Repair a broken pipeline

The snippet below reports a suspiciously perfect CV accuracy. It contains several distinct leaks. Find **all** of them, fix the code, and report the honest CV score.

Hints: look at the columns of the DataFrame, at where preprocessing is fit, and at what GridSearchCV actually sees.

In [7]:
rng = np.random.RandomState(42)
n = 400
X_clean = rng.randn(n, 10)
y = (X_clean[:, 0] + 0.3 * rng.randn(n) > 0).astype(int)

X_df = pd.DataFrame(X_clean, columns=["f{}".format(i) for i in range(10)])
X_df["id"] = np.arange(n)                 # row identifier
X_df["leak"] = y + rng.normal(0, 0.05, n)  # constructed from the target

# BUGGY WORKFLOW - find every leak
features = ["f{}".format(i) for i in range(10)] + ["leak", "id"]

scaler = StandardScaler().fit(X_df[features])
X_scaled = scaler.transform(X_df[features])
selector = SelectKBest(f_classif, k=8).fit(X_scaled, y)
X_sel = selector.transform(X_scaled)

gs_buggy = GridSearchCV(LogisticRegression(max_iter=3000), {"C": [0.01, 0.1, 1.0]},
                        cv=StratifiedKFold(5, shuffle=True, random_state=42))
gs_buggy.fit(X_sel, y)
print("BUGGY CV accuracy:", gs_buggy.best_score_)

BUGGY CV accuracy: 1.0


**Your fix.** Rewrite the workflow so that the reported score is honest: drop the leaky and useless columns, and put every preprocessing step inside a Pipeline used by GridSearchCV.


In [8]:
# Your code

**Interpretation.** Report the honest CV score and list every leak you found, explaining in one sentence each how it inflated the estimate.


**Your answer:**

## Takeaways

- Choose the validation scheme that matches the data-generating process: i.i.d. -> StratifiedKFold, grouped -> GroupKFold, time-ordered -> TimeSeriesSplit.
- Plain KFold on imbalanced data can produce folds without any positive samples, distorting both training and evaluation.
- Any preprocessing that uses validation-fold information (statistics or labels) before the split is leakage; the CV score becomes optimistic and can mislead model selection.
- GridSearchCV around a Pipeline refits every preprocessing step inside each training fold - this is the default way to tune a model.
- An optimistic CV score is not harmless: it creates wrong expectations for deployment and can select the wrong model.
- Nested CV (or a held-out test set used exactly once) is the protocol for an unbiased estimate of a tuned model.